# Basics

This notebook demonstrates four foundational security goals in communication systems:

| Goal                  | Description                                                       |
| --------------------- | ----------------------------------------------------------------- |
| **Confidentiality**   | Only the intended sender and receiver can understand the message. |
| **Authentication**    | Both parties can verify each other's identity.                    |
| **Message Integrity** | Any modification to the message in transit is detectable.         |
| **Availability**      | Services remain accessible to legitimate users.                   |

## Imports

In [76]:
import numpy as np

# Confidentiality

Two common approaches provide confidentiality: symmetric and asymmetric encryption.
Symmetric encryption is fast but requires secure key exchange.
Asymmetric encryption is slower but simplifies key distribution.

## Symmetric Encryption

### Custom Implementation

In [77]:
def apply_permutation_cipher(input_arr: np.ndarray, key: list) -> np.ndarray:
    """
    Encrypts an array using a block permutation cipher.
    Example key for your logic: [2, 3, 0, 1]
    """
    output = input_arr.copy()
    block_size = len(key)
    
    # Pad the array so we don't leave a raw, unencrypted remainder
    remainder = len(output) % block_size
    if remainder != 0:
        padding_length = block_size - remainder
        # Pad with zeros (or a random noise value for better security)
        output = np.pad(output, (0, padding_length), mode='constant')

    # Vectorized block shuffling
    blocks = output.reshape(-1, block_size)
    blocks[:, :] = blocks[:, key]
    
    return output

In [78]:
def str_to_ascii(text: str) -> np.ndarray:
    return np.array([ord(char) for char in text])

def ascii_to_str(ascii_codes: np.ndarray) -> str:
    s = ''.join(chr(int(code)) for code in ascii_codes)
    return s

In [79]:
def apply_function_to_text(text: str, algorithm) -> str:
    ascii_text = str_to_ascii(text)
    converted_array = algorithm(ascii_text)
    converted_text = ascii_to_str(converted_array)

    return converted_text

In [80]:
def xor_algorithm(key: int):
    def algo(arr: np.ndarray) -> np.ndarray:
        return arr ^ key
    return algo

In [81]:
def permutation_algorithm(key: list):
    def algo(arr: np.ndarray) -> np.ndarray:
        return apply_permutation_cipher(arr, key)
    return algo

In [82]:
def combined_algorithm(key: int):
    def algo(arr: np.ndarray) -> np.ndarray:
        xored = arr ^ key
        shuffled = apply_permutation_cipher(xored, [2, 3, 0, 1])
        return shuffled
    return algo

In [83]:
def combined_algorithm2(key: int):
    def algo(arr: np.ndarray) -> np.ndarray:
        xored = arr ^ key
        shuffled = apply_permutation_cipher(xored, [1, 0, 3, 2])
        return shuffled
    return algo

In [84]:
text = "This is some sample Boi Text"

# Compare all algorithms
print("[Custom Symmetric] Original plaintext:", repr(text))

# 1. Permutation only
algorithm1 = permutation_algorithm([2, 3, 0, 1])
encrypted1 = apply_function_to_text(text=text, algorithm=algorithm1)
print("[Custom Symmetric] Permutation [2,3,0,1] ciphertext:", repr(encrypted1))

# 2. XOR only
algorithm2 = xor_algorithm(42)
encrypted2 = apply_function_to_text(text=text, algorithm=algorithm2)
print("[Custom Symmetric] XOR (key=42) ciphertext:", repr(encrypted2))

# 3. Combined (XOR + Permutation [2,3,0,1])
algorithm3 = combined_algorithm(42)
encrypted3 = apply_function_to_text(text=text, algorithm=algorithm3)
print("[Custom Symmetric] XOR+Permutation [2,3,0,1] ciphertext:", repr(encrypted3))

# 4. Combined with [1,0,3,2]
algorithm4 = combined_algorithm2(42)
encrypted4 = apply_function_to_text(text=text, algorithm=algorithm4)
print("[Custom Symmetric] XOR+Permutation [1,0,3,2] ciphertext:", repr(encrypted4))

# Verify decryption with one of them
decrypted = apply_function_to_text(text=encrypted3, algorithm=algorithm3)
print("[Custom Symmetric] Round-trip decrypted plaintext:", repr(decrypted))

[Custom Symmetric] Original plaintext: 'This is some sample Boi Text'
[Custom Symmetric] Permutation [2,3,0,1] ciphertext: 'isThs  imesoam se pli BoxtTe'
[Custom Symmetric] XOR (key=42) ciphertext: '~BCY\nCY\nYEGO\nYKGZFO\nhEC\n~OR^'
[Custom Symmetric] XOR+Permutation [2,3,0,1] ciphertext: 'CY~BY\n\nCGOYEKG\nYO\nZFC\nhER^~O'
[Custom Symmetric] XOR+Permutation [1,0,3,2] ciphertext: 'B~YCC\n\nYEYOGY\nGKFZ\nOEh\nCO~^R'
[Custom Symmetric] Round-trip decrypted plaintext: 'This is some sample Boi Text'


### Standard with Cryptography

In [85]:
from cryptography.fernet import Fernet

# 1. Generate the symmetric key
key = Fernet.generate_key()

# Initialize the Fernet cipher with the key
cipher = Fernet(key)

# --- Encryption/Decryption Process ---
message = b"Hallo Hier"

# 2. Encrypt the message
ciphertext = cipher.encrypt(message)

print(f"[Fernet] Symmetric key (keep secret): {key.decode('utf-8')}")
print(f"[Fernet] Ciphertext preview: {ciphertext[:40]}... (truncated)")

# 3. Decrypt the message
plaintext = cipher.decrypt(ciphertext)

print(f"[Fernet] Decrypted plaintext: {plaintext.decode('utf-8')}")

[Fernet] Symmetric key (keep secret): Mb9VYJrYSGcINVekfdeR2kClQNI_ipR7znL35n9djfU=
[Fernet] Ciphertext preview: b'gAAAAABpwYsNIM6soeEaQ5CcUTzxr6l53b1yFyT6'... (truncated)
[Fernet] Decrypted plaintext: Hallo Hier


## Asymmetric Encryption

In [86]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes

# 1. Generate the private key
# 65537 is the standard public exponent. 2048 is a secure, standard key size.
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# 2. Extract the public key (to share with the sender)
public_key = private_key.public_key()

# --- Encryption/Decryption Process ---
message = b"Hallo Hier"
print(f"[RSA] Original message bytes: {message}")

# 3. Encrypt the message using the public key
# OAEP padding with SHA256 is the modern, secure standard for RSA encryption.
ciphertext = public_key.encrypt(
    message,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"[RSA] Ciphertext preview: {ciphertext[:20]}... (truncated)")

# 4. Decrypt the message using the private key
plaintext = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(f"[RSA] Decrypted plaintext: {plaintext.decode('utf-8')}")

[RSA] Original message bytes: b'Hallo Hier'
[RSA] Ciphertext preview: b'[@\x12M>\x05\xd9\x13E\x83@\x85A\xd8\x18k\x91\xff[N'... (truncated)
[RSA] Decrypted plaintext: Hallo Hier


## Authentication

Authentication ensures both parties can confirm who they are communicating with.
A common approach is a challenge-response flow using a nonce (a one-time random value).

In [87]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
from cryptography.fernet import Fernet
import base64
import json
import random

SIGN_PADDING = padding.PSS(
    mgf=padding.MGF1(hashes.SHA256()),
    salt_length=padding.PSS.MAX_LENGTH
)

KEY_EXCHANGE_PADDING = padding.OAEP(
    mgf=padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(),
    label=None
)

class Party:
    def __init__(self) -> None:
        self.private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        self.public_key = self.private_key.public_key()

    def generate_nonce(self) -> int:
        return random.randint(0, int(10e4))

    def _hash_bytes(self, data: bytes) -> bytes:
        digest = hashes.Hash(hashes.SHA256())
        digest.update(data)
        return digest.finalize()

    def _sign_bytes(self, data: bytes) -> bytes:
        return self.private_key.sign(
            data,
            SIGN_PADDING,
            hashes.SHA256()
        )

    def _verify_signature(self, data: bytes, signature: bytes, signer_public_key) -> None:
        signer_public_key.verify(
            signature,
            data,
            SIGN_PADDING,
            hashes.SHA256()
        )

    def sign_nonce(self, nonce: int) -> bytes:
        nonce_bytes = str(nonce).encode("utf-8")
        return self._sign_bytes(nonce_bytes)

    def verify_nonce_signature(self, nonce: int, signature: bytes, signer_public_key) -> None:
        nonce_bytes = str(nonce).encode("utf-8")
        self._verify_signature(nonce_bytes, signature, signer_public_key)

    def sign_message_hash(self, message: bytes) -> bytes:
        message_hash = self._hash_bytes(message)
        return self._sign_bytes(message_hash)

    def verify_message_hash_signature(self, message: bytes, signature: bytes, signer_public_key) -> bool:
        message_hash = self._hash_bytes(message)
        try:
            self._verify_signature(message_hash, signature, signer_public_key)
            return True
        except Exception:
            return False

    def generate_session_key(self) -> bytes:
        return Fernet.generate_key()

    def encrypt_session_key_for_peer(self, session_key: bytes, peer_public_key) -> bytes:
        return peer_public_key.encrypt(session_key, KEY_EXCHANGE_PADDING)

    def decrypt_session_key_from_peer(self, encrypted_session_key: bytes) -> bytes:
        return self.private_key.decrypt(encrypted_session_key, KEY_EXCHANGE_PADDING)

    def create_message_bundle(self, message: bytes) -> bytes:
        signature_over_hash = self.sign_message_hash(message)
        bundle = {
            "message": base64.b64encode(message).decode("utf-8"),
            "signed_hash": base64.b64encode(signature_over_hash).decode("utf-8")
        }
        return json.dumps(bundle).encode("utf-8")

    def encrypt_bundle_with_session_key(self, bundle: bytes, session_key: bytes) -> bytes:
        cipher = Fernet(session_key)
        return cipher.encrypt(bundle)

    def decrypt_bundle_with_session_key(self, encrypted_bundle: bytes, session_key: bytes) -> bytes:
        cipher = Fernet(session_key)
        return cipher.decrypt(encrypted_bundle)

    def parse_message_bundle(self, bundle: bytes) -> tuple[bytes, bytes]:
        parsed = json.loads(bundle.decode("utf-8"))
        message = base64.b64decode(parsed["message"])
        signature_over_hash = base64.b64decode(parsed["signed_hash"])
        return message, signature_over_hash

In [88]:
partyA = Party()
partyB = Party()

print("[Authentication] Step 1: A claims identity to B.")

# Step 2: B sends a nonce challenge to A
nonce_from_b_to_a = partyB.generate_nonce()
print(f"[Authentication] Step 2: B sends nonce challenge: {nonce_from_b_to_a}")

# Step 3: A signs the nonce with A's private key and sends the signature to B
signed_nonce = partyA.sign_nonce(nonce_from_b_to_a)
print("[Authentication] Step 3: A sends signed nonce to B.")

# Step 4: B obtains A's public key
public_key_a = partyA.public_key
print("[Authentication] Step 4: B obtains A's public key.")

# Step 5: B verifies the signature over the original nonce
try:
    partyB.verify_nonce_signature(nonce_from_b_to_a, signed_nonce, public_key_a)
    print("[Authentication] Step 5: Verification successful. A is authenticated.")
except Exception:
    print("[Authentication] Step 5: Verification failed. A is not authenticated.")

[Authentication] Step 1: A claims identity to B.
[Authentication] Step 2: B sends nonce challenge: 76715
[Authentication] Step 3: A sends signed nonce to B.
[Authentication] Step 4: B obtains A's public key.
[Authentication] Step 5: Verification successful. A is authenticated.


## Message Integrity (Hash + Signature)

For **integrity and sender authentication only**, the message is not encrypted.
The sender transmits the plaintext message `m` together with a signature over `H(m)`.

**SENDER (Party A)**

$$
m \rightarrow \left\{ \begin{array}{l} 
m \\ 
K_A^-(H(m)) 
\end{array} \right\} \rightarrow m, K_A^-(H(m))
$$

**RECEIVER (Party B)**

$$
m, K_A^-(H(m)) \rightarrow \left\{ \begin{array}{l} 
m \rightarrow H(m) \\ 
K_A^+(K_A^-(H(m)))=H(m) \\
m
\end{array} \right\} \rightarrow H(m)\stackrel{?}{=}H(m),m
$$

If the two hashes match, the message integrity holds and the signature confirms it came from Party A.

In [89]:
# Integrity + authentication demo (message remains plaintext)
partyA = Party()
partyB = Party()

original_message = b"Transfer CHF 100 to account 12345"
signature_over_hash = partyA.sign_message_hash(original_message)
public_key_a = partyA.public_key

print(f"[Integrity] Sender transmits plaintext message: {original_message.decode('utf-8')}")
print(f"[Integrity] Sender transmits signed hash preview: {signature_over_hash[:20]}... (truncated)")

is_valid = partyB.verify_message_hash_signature(
    original_message,
    signature_over_hash,
    public_key_a
)
print(f"[Integrity] Receiver verification result (untampered): {is_valid}")

tampered_message = b"Transfer CHF 9000 to account 12345"
is_valid_tampered = partyB.verify_message_hash_signature(
    tampered_message,
    signature_over_hash,
    public_key_a
)
print(f"[Integrity] Receiver verification result (tampered): {is_valid_tampered}")

[Integrity] Sender transmits plaintext message: Transfer CHF 100 to account 12345
[Integrity] Sender transmits signed hash preview: b'q!\x08\xa8O\x1d\xeb\x04\xc0t0\xa4!\x1f#\x83F\x88\x0e\x8a'... (truncated)
[Integrity] Receiver verification result (untampered): True
[Integrity] Receiver verification result (tampered): False


## Everything Together (Confidentiality + Integrity + Authentication)

This flow combines all three goals:

1. **Asymmetric key exchange**: Party A generates a session key `K_S` and encrypts it with Party B's public key.
2. **Integrity + authentication**: Party A signs `H(m)` with Party A's private key.
3. **Confidential transport**: Party A encrypts the message bundle `(m, K_A^-(H(m)))` with `K_S`.

**SENDER**

$$
\text{Party A generates } K_S \rightarrow K_B^+(K_S) \rightarrow \text{send to Party B}
$$

**RECEIVER**

$$
\text{Party B receives } K_B^+(K_S) \rightarrow K_B^-(K_B^+(K_S)) \rightarrow \text{both parties have } K_S
$$

**SENDER (Party A)**

$$
m \rightarrow \left\{ \begin{array}{l}
m \\
K_A^-(H(m))
\end{array} \right\} \rightarrow \text{bundle } (m, K_A^-(H(m))) \rightarrow K_S(m, K_A^-(H(m)))
$$

**RECEIVER (Party B)**

$$
K_S(m, K_A^-(H(m))) \rightarrow \text{decrypt with } K_S \rightarrow m, K_A^-(H(m)) \rightarrow \left\{ \begin{array}{l}
m \rightarrow H(m) \\
K_A^+(K_A^-(H(m)))=H(m) \\
m
\end{array} \right\} \rightarrow H(m)\stackrel{?}{=}H(m),m
$$

In [90]:
# Everything together demo: confidentiality + integrity + authentication
partyA = Party()
partyB = Party()

# Step 1: Party A generates a fresh session key
session_key_a = partyA.generate_session_key()
print("[Everything] Step 1: A generates session key K_S.")

# Step 2: Party A encrypts K_S with Party B's public key and sends it
encrypted_session_key_for_b = partyA.encrypt_session_key_for_peer(session_key_a, partyB.public_key)
print(f"[Everything] Step 2: A sends encrypted session key preview: {encrypted_session_key_for_b[:20]}... (truncated)")

# Step 3: Party B decrypts K_S with B's private key
session_key_b = partyB.decrypt_session_key_from_peer(encrypted_session_key_for_b)
print(f"[Everything] Step 3: B recovered same K_S: {session_key_b == session_key_a}")

# Step 4: Party A creates (m, K_A^-(H(m)))
message = b"Meet at 10:00 near the main station."
bundle = partyA.create_message_bundle(message)
print("[Everything] Step 4: A creates signed message-hash bundle.")

# Step 5: Party A encrypts the bundle with K_S and sends it
encrypted_bundle = partyA.encrypt_bundle_with_session_key(bundle, session_key_a)
print(f"[Everything] Step 5: A sends encrypted bundle preview: {encrypted_bundle[:30]}... (truncated)")

# Step 6: Party B decrypts bundle with K_S
decrypted_bundle = partyB.decrypt_bundle_with_session_key(encrypted_bundle, session_key_b)
received_message, received_signed_hash = partyB.parse_message_bundle(decrypted_bundle)
print(f"[Everything] Step 6: B recovered plaintext message: {received_message.decode('utf-8')}")

# Step 7: Party B verifies integrity + authentication using A's public key
verified = partyB.verify_message_hash_signature(
    received_message,
    received_signed_hash,
    partyA.public_key
)
print(f"[Everything] Step 7: Integrity/authentication verification result (untampered): {verified}")

# Step 8: Demonstrate tampering detection after decryption
tampered_received_message = b"Meet at 22:00 near the main station."
verified_tampered = partyB.verify_message_hash_signature(
    tampered_received_message,
    received_signed_hash,
    partyA.public_key
)
print(f"[Everything] Step 8: Integrity/authentication verification result (tampered): {verified_tampered}")

[Everything] Step 1: A generates session key K_S.
[Everything] Step 2: A sends encrypted session key preview: b'"\x8b\xb5$&\x80la\xca\xd3\x8e$d\xe7[DUO\x95\xbd'... (truncated)
[Everything] Step 3: B recovered same K_S: True
[Everything] Step 4: A creates signed message-hash bundle.
[Everything] Step 5: A sends encrypted bundle preview: b'gAAAAABpwYsN60HBzxZTbaRMVDIUvz'... (truncated)
[Everything] Step 6: B recovered plaintext message: Meet at 10:00 near the main station.
[Everything] Step 7: Integrity/authentication verification result (untampered): True
[Everything] Step 8: Integrity/authentication verification result (tampered): False
